In [3]:
## load seq_22612 mutagenesis library + DeepSHAP attributions
## (25k mutants, DeepSHAP w/ 100 dinuc-shuffled background, DeepSTARR Dev task)
## attributions are gradient-corrected (mean-centered across channels) on load.

import h5py
import numpy as np
import pandas as pd

SEQ_LABEL  = '22612'
DATA_DIR   = '/grid/koo/home/pmantill/projects/VBI/init_impliment/data/seq_22612'
LIB_H5     = f'{DATA_DIR}/library_25000.h5'
ATTR_H5    = f'{DATA_DIR}/attributions_deepshap_25000.h5'
NEUTRAL_CSV = '/grid/koo/home/pmantill/projects/Motif_Swap_Experiments/Replicating_Motif_Swap_Manuscript/experimental_library_generation/Binned_libraries/Dev_full_data/mid_df.csv'

def grad_correct(attr):
    """Hypothetical -> grad-corrected: mean-center across the 4-channel axis."""
    return attr - attr.mean(axis=-1, keepdims=True)

with h5py.File(LIB_H5, 'r') as f:
    x_mut = f['sequences'][:]
    y_mut = f['predictions'][:]
print(f'library: x_mut {x_mut.shape}  y_mut {y_mut.shape}  (WT at index 0)')

with h5py.File(ATTR_H5, 'r') as f:
    attributions = grad_correct(f['attributions'][:])
print(f'attributions (grad-corrected): {attributions.shape}')

wt_seq  = x_mut[0]
wt_attr = attributions[0]

neutral_df = pd.read_csv(NEUTRAL_CSV, usecols=['test_idx', 'str_seq'])
print(f'neutral bin: {len(neutral_df)} sequences')

ALPHA_MAP = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
def str_to_onehot(seq_str):
    ohe = np.zeros((len(seq_str), 4), dtype=np.float32)
    for j, b in enumerate(seq_str):
        if b in ALPHA_MAP:
            ohe[j, ALPHA_MAP[b]] = 1.0
    return ohe

def pick_neutral(seed=None):
    """Random row from the neutral bin. Returns (test_idx, str_seq, one_hot)."""
    rng = np.random.default_rng(seed)
    i = int(rng.integers(len(neutral_df)))
    row = neutral_df.iloc[i]
    return int(row['test_idx']), row['str_seq'], str_to_onehot(row['str_seq'])

idx, seq_str, seq_ohe = pick_neutral(seed=0)
print(f'neutral pick: test_idx={idx}  ohe {seq_ohe.shape}')

library: x_mut (25000, 249, 4)  y_mut (25000, 1)  (WT at index 0)
attributions (grad-corrected): (25000, 249, 4)
neutral bin: 3000 sequences
neutral pick: test_idx=18511  ohe (249, 4)


In [6]:
## load bad-case neutral seqs (test_idx 21795, 18511)
## each entry: str_seq, ohe (249,4), attr (Scaled_foreground, grad-corrected)

import re
import numpy as np
import pandas as pd

MID_CSV = '/grid/koo/home/pmantill/projects/Motif_Swap_Experiments/Replicating_Motif_Swap_Manuscript/experimental_library_generation/Binned_libraries/Dev_full_data/mid_df.csv'
BAD_IDS = [21795, 18511]

mid_idx = pd.read_csv(MID_CSV, usecols=['test_idx', 'str_seq', 'real_score', 'evoaug_pred'])

_FLOAT_RE = re.compile(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?')
def parse_tensor_str(s, L=249, A=4):
    vals = _FLOAT_RE.findall(s)
    arr = np.array(vals, dtype=np.float32)
    assert arr.size == L * A, f'got {arr.size} floats, expected {L*A}'
    return arr.reshape(L, A)

def fetch_tensor(row_i, col='Scaled_foreground'):
    sub = pd.read_csv(MID_CSV, usecols=[col],
                      skiprows=lambda r: r != 0 and r != row_i + 1)
    return grad_correct(parse_tensor_str(sub[col].iloc[0]))

bad_cases = {}
for tid in BAD_IDS:
    hits = mid_idx.index[mid_idx['test_idx'] == tid].tolist()
    assert len(hits) == 1, f'test_idx {tid}: {len(hits)} matches'
    row_i = hits[0]
    row = mid_idx.iloc[row_i]
    bad_cases[tid] = {
        'str_seq':     row['str_seq'],
        'ohe':         str_to_onehot(row['str_seq']),
        'attr':        fetch_tensor(row_i, 'Scaled_foreground'),
        'real_score':  float(row['real_score']),
        'evoaug_pred': float(row['evoaug_pred']),
    }
    print(f'bad test_idx={tid}  real={row["real_score"]:.3f}  pred={row["evoaug_pred"]:.3f}  attr {bad_cases[tid]["attr"].shape}')

bad test_idx=21795  real=3.439  pred=0.304  attr (249, 4)
bad test_idx=18511  real=0.060  pred=0.115  attr (249, 4)
